In [2]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder

In [3]:
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('../../data/laptop_data.csv')
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,96095.8080


### Check the dimensions of dataframe

In [5]:
df.shape

(1303, 12)

### Check whether there are missing values in dataset

In [6]:
df.isna().sum()

Unnamed: 0          0
Company             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price               0
dtype: int64

### Check whether there are duplicate instances in dataframe

In [7]:
df.loc[df.duplicated()]

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price


### Check the overall details about the dataframe

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   TypeName          1303 non-null   object 
 3   Inches            1303 non-null   float64
 4   ScreenResolution  1303 non-null   object 
 5   Cpu               1303 non-null   object 
 6   Ram               1303 non-null   object 
 7   Memory            1303 non-null   object 
 8   Gpu               1303 non-null   object 
 9   OpSys             1303 non-null   object 
 10  Weight            1303 non-null   object 
 11  Price             1303 non-null   float64
dtypes: float64(2), int64(1), object(9)
memory usage: 122.3+ KB


### Print all object type variables

In [9]:
print([x for x in df.columns if df[x].dtype == 'object'])

['Company', 'TypeName', 'ScreenResolution', 'Cpu', 'Ram', 'Memory', 'Gpu', 'OpSys', 'Weight']


### Print all int and float type variables

In [10]:
print([x for x in df.columns if df[x].dtype in ('int64', 'float64')])

['Unnamed: 0', 'Inches', 'Price']


### Remove the text 'GB' from each value in the Ram column and convert the result to 64-bit integers

In [11]:
df['Ram'] = df['Ram'].str.replace('GB', '').astype('int64')

### Remove the text 'kg' from each value in the Weight column and convert the result to 64-bit floats

In [12]:
df['Weight'] = df['Weight'].str.replace('kg', '').astype('float64')

In [13]:
df.head(5)

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,71378.6832
1,1,Apple,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,47895.5232
2,2,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,30636.0000
3,3,Apple,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,135195.3360
4,4,Apple,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,96095.8080


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        1303 non-null   int64  
 1   Company           1303 non-null   object 
 2   TypeName          1303 non-null   object 
 3   Inches            1303 non-null   float64
 4   ScreenResolution  1303 non-null   object 
 5   Cpu               1303 non-null   object 
 6   Ram               1303 non-null   int64  
 7   Memory            1303 non-null   object 
 8   Gpu               1303 non-null   object 
 9   OpSys             1303 non-null   object 
 10  Weight            1303 non-null   float64
 11  Price             1303 non-null   float64
dtypes: float64(3), int64(2), object(7)
memory usage: 122.3+ KB


### Fetches the live exchange rate and converts the DataFrame's price column to LKR

In [15]:
import requests
import pandas as pd

def get_exchange_rate():
    response = requests.get('https://api.exchangerate-api.com/v4/latest/INR')
    data = response.json()
    return data['rates']['LKR']

exchange_rate = get_exchange_rate()
print(f"Current exchange rate: 1 INR = {exchange_rate} LKR")

df['Price'] = df['Price'] * exchange_rate
pd.set_option('display.float_format', '{:,.2f}'.format)

df.head()

Current exchange rate: 1 INR = 3.51 LKR


,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,"250,539.18"
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,"168,113.29"
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,"107,532.36"
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,"474,535.63"
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,"337,296.29"


### Counts the number of times each unique laptop brand

In [16]:
df['Company'].value_counts()

Company
Dell         297
Lenovo       297
HP           274
Asus         158
Acer         103
MSI           54
Toshiba       48
Apple         21
Samsung        9
Mediacom       7
Razer          7
Microsoft      6
Vero           4
Xiaomi         4
Chuwi          3
Fujitsu        3
Google         3
LG             3
Huawei         2
Name: count, dtype: int64

### Defined a function to return the brand 'Other' if count less than 10

In [17]:
def add_company(inpt):
    other_brands = {
        'Samsung', 'Razer', 'Mediacom', 'Microsoft', 'Xiaomi', 'Vero',
        'Chuwi', 'Google', 'Fujitsu', 'LG', 'Huawei'
    }
    return 'Other' if inpt in other_brands else inpt

In [18]:
df['Company'] = df['Company'].apply(add_company)

In [19]:
df['Company'].value_counts()

Company
Dell       297
Lenovo     297
HP         274
Asus       158
Acer       103
MSI         54
Other       51
Toshiba     48
Apple       21
Name: count, dtype: int64

In [20]:
df['TypeName'].value_counts()

TypeName
Notebook              727
Gaming                205
Ultrabook             196
2 in 1 Convertible    121
Workstation            29
Netbook                25
Name: count, dtype: int64

In [21]:
len(df['TypeName'].value_counts())

6

In [22]:
df['ScreenResolution'].value_counts()

ScreenResolution
Full HD 1920x1080                                507
1366x768                                         281
IPS Panel Full HD 1920x1080                      230
IPS Panel Full HD / Touchscreen 1920x1080         53
Full HD / Touchscreen 1920x1080                   47
1600x900                                          23
Touchscreen 1366x768                              16
Quad HD+ / Touchscreen 3200x1800                  15
IPS Panel 4K Ultra HD 3840x2160                   12
IPS Panel 4K Ultra HD / Touchscreen 3840x2160     11
4K Ultra HD / Touchscreen 3840x2160               10
IPS Panel 1366x768                                 7
Touchscreen 2560x1440                              7
4K Ultra HD 3840x2160                              7
IPS Panel Retina Display 2304x1440                 6
IPS Panel Retina Display 2560x1600                 6
Touchscreen 2256x1504                              6
IPS Panel Quad HD+ / Touchscreen 3200x1800         6
IPS Panel Touchscreen 2560x14

### Created new binary columns to identify specific display features

In [23]:
df['Touchscreen'] = df['ScreenResolution'].apply(lambda x: 1 if 'Touchscreen' in x else 0)
df['Ips'] = df['ScreenResolution'].apply(lambda x: 1 if 'IPS' in x else 0)
df['HD'] = df['ScreenResolution'].apply(lambda x: 1 if 'HD' in x else 0)

In [24]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Touchscreen,Ips,HD
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,"250,539.18",0,1,0
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,"168,113.29",0,0,0
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,"107,532.36",0,0,1
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,"474,535.63",0,1,0
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,"337,296.29",0,1,0


In [25]:
df['Memory'].value_counts()

Memory
256GB SSD                        412
1TB HDD                          223
500GB HDD                        132
512GB SSD                        118
128GB SSD +  1TB HDD              94
128GB SSD                         76
256GB SSD +  1TB HDD              73
32GB Flash Storage                38
2TB HDD                           16
64GB Flash Storage                15
1TB SSD                           14
512GB SSD +  1TB HDD              14
256GB SSD +  2TB HDD              10
1.0TB Hybrid                       9
256GB Flash Storage                8
16GB Flash Storage                 7
32GB SSD                           6
180GB SSD                          5
128GB Flash Storage                4
16GB SSD                           3
512GB SSD +  2TB HDD               3
128GB SSD +  2TB HDD               2
256GB SSD +  256GB SSD             2
512GB Flash Storage                2
1TB SSD +  1TB HDD                 2
256GB SSD +  500GB HDD             2
64GB SSD                       

### Split memory from '+' to Storage_1 and Storage_2

In [26]:
df[['Storage_1', 'Storage_2']] = df['Memory'].str.split('+', expand=True)

In [27]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Touchscreen,Ips,HD,Storage_1,Storage_2
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37,"250,539.18",0,1,0,128GB SSD,None
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34,"168,113.29",0,0,0,128GB Flash Storage,None
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,"107,532.36",0,0,1,256GB SSD,None
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,1.83,"474,535.63",0,1,0,512GB SSD,None
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37,"337,296.29",0,1,0,256GB SSD,None


In [28]:
df.tail()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price,Touchscreen,Ips,HD,Storage_1,Storage_2
1298,1298,Lenovo,2 in 1 Convertible,14.00,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4,128GB SSD,Intel HD Graphics 520,Windows 10,1.80,"119,314.17",1,1,1,128GB SSD,None
1299,1299,Lenovo,2 in 1 Convertible,13.30,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16,512GB SSD,Intel HD Graphics 520,Windows 10,1.30,"280,332.19",1,1,1,512GB SSD,None
1300,1300,Lenovo,Notebook,14.00,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2,64GB Flash Storage,Intel HD Graphics,Windows 10,1.50,"42,825.93",0,0,0,64GB Flash Storage,None
1301,1301,HP,Notebook,15.60,1366x768,Intel Core i7 6500U 2.5GHz,6,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19,"142,877.78",0,0,0,1TB HDD,None
1302,1302,Asus,Notebook,15.60,1366x768,Intel Celeron Dual Core N3050 1.6GHz,4,500GB HDD,Intel HD Graphics,Windows 10,2.20,"69,007.72",0,0,0,500GB HDD,None


### Defined a function split Storage_1 and Storage_2 to another subset of columns

In [29]:
def extract_storage_info(entry):
    if pd.isnull(entry) or not isinstance(entry, str) or entry.strip() == '':
        return pd.Series([None, None])
    
    entry = entry.strip()
    size = None
    
    # Convert TB or GB
    if 'TB' in entry:
        size = int(float(''.join(filter(lambda x: x.isdigit() or x == '.', entry))) * 1000)
    elif 'GB' in entry:
        size = int(''.join(filter(lambda x: x.isdigit(), entry)))
    
    # Identify type
    if 'SSD' in entry:
        storage_type = 'SSD'
    elif 'HDD' in entry:
        storage_type = 'HDD'
    elif 'Flash' in entry:
        storage_type = 'Flash Storage'
    elif 'Hybrid' in entry:
        storage_type = 'Hybrid'
    else:
        storage_type = 'Unknown'
    
    return pd.Series([storage_type, size])


### Applied the above function to columns Storage_1 and Storage_2 seperately

In [30]:
df[['Storage_1_Type', 'Storage_1_Size_GB']] = df['Storage_1'].apply(extract_storage_info)
df[['Storage_2_Type', 'Storage_2_Size_GB']] = df['Storage_2'].apply(extract_storage_info)

In [31]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,...,Price,Touchscreen,Ips,HD,Storage_1,Storage_2,Storage_1_Type,Storage_1_Size_GB,Storage_2_Type,Storage_2_Size_GB
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,...,"250,539.18",0,1,0,128GB SSD,None,SSD,128,None,NaN
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,...,"168,113.29",0,0,0,128GB Flash Storage,None,Flash Storage,128,None,NaN
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,...,"107,532.36",0,0,1,256GB SSD,None,SSD,256,None,NaN
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,...,"474,535.63",0,1,0,512GB SSD,None,SSD,512,None,NaN
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,...,"337,296.29",0,1,0,256GB SSD,None,SSD,256,None,NaN


In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1303 non-null   int64  
 1   Company            1303 non-null   object 
 2   TypeName           1303 non-null   object 
 3   Inches             1303 non-null   float64
 4   ScreenResolution   1303 non-null   object 
 5   Cpu                1303 non-null   object 
 6   Ram                1303 non-null   int64  
 7   Memory             1303 non-null   object 
 8   Gpu                1303 non-null   object 
 9   OpSys              1303 non-null   object 
 10  Weight             1303 non-null   float64
 11  Price              1303 non-null   float64
 12  Touchscreen        1303 non-null   int64  
 13  Ips                1303 non-null   int64  
 14  HD                 1303 non-null   int64  
 15  Storage_1          1303 non-null   object 
 16  Storage_2          208 n

### Replaced NaN with 0 value

In [33]:
df['Storage_2_Size_GB'] = df['Storage_2_Size_GB'].fillna(0).astype('Int64')

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1303 non-null   int64  
 1   Company            1303 non-null   object 
 2   TypeName           1303 non-null   object 
 3   Inches             1303 non-null   float64
 4   ScreenResolution   1303 non-null   object 
 5   Cpu                1303 non-null   object 
 6   Ram                1303 non-null   int64  
 7   Memory             1303 non-null   object 
 8   Gpu                1303 non-null   object 
 9   OpSys              1303 non-null   object 
 10  Weight             1303 non-null   float64
 11  Price              1303 non-null   float64
 12  Touchscreen        1303 non-null   int64  
 13  Ips                1303 non-null   int64  
 14  HD                 1303 non-null   int64  
 15  Storage_1          1303 non-null   object 
 16  Storage_2          208 n

In [35]:
df['Cpu'].value_counts()

Cpu
Intel Core i5 7200U 2.5GHz       190
Intel Core i7 7700HQ 2.8GHz      146
Intel Core i7 7500U 2.7GHz       134
Intel Core i7 8550U 1.8GHz        73
Intel Core i5 8250U 1.6GHz        72
                                ... 
Intel Core i5 7200U 2.70GHz        1
Intel Core M M7-6Y75 1.2GHz        1
Intel Core M 6Y54 1.1GHz           1
AMD E-Series 9000 2.2GHz           1
Samsung Cortex A72&A53 2.0GHz      1
Name: count, Length: 118, dtype: int64

### Extracts the first three words from the 'Cpu' column and created a new column

In [36]:
df['cpu_name'] = df['Cpu'].apply(lambda x:" ".join(x.split()[0:3]))

In [37]:
df['cpu_name'].value_counts()

cpu_name
Intel Core i7               527
Intel Core i5               423
Intel Core i3               136
Intel Celeron Dual           80
Intel Pentium Quad           27
Intel Core M                 19
AMD A9-Series 9420           12
AMD A6-Series 9220            8
Intel Celeron Quad            8
AMD A12-Series 9720P          7
Intel Atom x5-Z8350           5
AMD A8-Series 7410            4
Intel Atom x5-Z8550           4
AMD A9-Series 9410            3
Intel Pentium Dual            3
AMD Ryzen 1700                3
AMD A9-Series A9-9420         2
AMD E-Series E2-9000e         2
AMD A10-Series A10-9620P      2
AMD A6-Series A6-9220         2
AMD E-Series 7110             2
AMD A10-Series 9620P          2
AMD A10-Series 9600P          2
Intel Xeon E3-1505M           2
Intel Xeon E3-1535M           2
Intel Atom X5-Z8350           2
Intel Atom x5-Z8300           1
AMD E-Series 6110             1
AMD E-Series 9000e            1
AMD E-Series E2-6110          1
AMD FX 9830P                  1

### Defined a function to categories cpu names

In [38]:
def set_processor(name):
    if name == 'Intel Core i7' or name == 'Intel Core i5' or name == 'Intel Core i3' or name == 'Intel Celeron Dual':
        return name
    else:
        if name.split()[0] == 'AMD':
            return 'AMD'
        else:
            return 'Other'

In [39]:
df['cpu_name'] = df['cpu_name'].apply(set_processor)

In [40]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,...,Touchscreen,Ips,HD,Storage_1,Storage_2,Storage_1_Type,Storage_1_Size_GB,Storage_2_Type,Storage_2_Size_GB,cpu_name
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,...,0,1,0,128GB SSD,None,SSD,128,None,0,Intel Core i5
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,...,0,0,0,128GB Flash Storage,None,Flash Storage,128,None,0,Intel Core i5
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,...,0,0,1,256GB SSD,None,SSD,256,None,0,Intel Core i5
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,...,0,1,0,512GB SSD,None,SSD,512,None,0,Intel Core i7
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,...,0,1,0,256GB SSD,None,SSD,256,None,0,Intel Core i5


In [41]:
df['Gpu'].value_counts()

Gpu
Intel HD Graphics 620      281
Intel HD Graphics 520      185
Intel UHD Graphics 620      68
Nvidia GeForce GTX 1050     66
Nvidia GeForce GTX 1060     48
                          ... 
Nvidia Quadro M500M          1
AMD Radeon R7 M360           1
Nvidia Quadro M3000M         1
Nvidia GeForce 960M          1
ARM Mali T860 MP4            1
Name: count, Length: 110, dtype: int64

### Extracts the first two words from the 'Gpu' column and created a new column

In [42]:
df['gpu_name'] = df['Gpu'].apply(lambda x:" ".join(x.split()[0:2]))

In [43]:
df['gpu_name'].value_counts()

gpu_name
Intel HD          639
Nvidia GeForce    368
AMD Radeon        173
Intel UHD          68
Nvidia Quadro      31
Intel Iris         14
AMD FirePro         5
AMD R4              1
AMD R17M-M1-70      1
Nvidia GTX          1
Intel Graphics      1
ARM Mali            1
Name: count, dtype: int64

### Defined a function to categories gpu names

In [44]:
def set_gpu(name):
    if name == 'Intel HD' or name == 'Nvidia GeForce' or name == 'Nvidia Quadro' or name == 'Intel UHD':
        return name
    else:
        if name.split()[0] == 'AMD':
            return 'AMD'
        else:
            return 'Other'

In [45]:
df['gpu_name'] = df['gpu_name'].apply(set_gpu)

In [46]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,...,Ips,HD,Storage_1,Storage_2,Storage_1_Type,Storage_1_Size_GB,Storage_2_Type,Storage_2_Size_GB,cpu_name,gpu_name
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,macOS,...,1,0,128GB SSD,None,SSD,128,None,0,Intel Core i5,Other
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,macOS,...,0,0,128GB Flash Storage,None,Flash Storage,128,None,0,Intel Core i5,Intel HD
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,...,0,1,256GB SSD,None,SSD,256,None,0,Intel Core i5,Intel HD
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,macOS,...,1,0,512GB SSD,None,SSD,512,None,0,Intel Core i7,AMD
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,macOS,...,1,0,256GB SSD,None,SSD,256,None,0,Intel Core i5,Other


In [47]:
df['OpSys'].value_counts()

OpSys
Windows 10      1072
No OS             66
Linux             62
Windows 7         45
Chrome OS         27
macOS             13
Mac OS X           8
Windows 10 S       8
Android            2
Name: count, dtype: int64

### Defined a function to categories OS names

In [48]:
def set_OpSys(name):
    if name == 'Windows 10' or name == 'No OS' or name == 'Linux' or name == 'Windows 7':
        return name
    else:
        return 'Other'

In [49]:
df['OpSys'] = df['OpSys'].apply(set_OpSys)

In [50]:
df.head()

,Unnamed: 0,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,...,Ips,HD,Storage_1,Storage_2,Storage_1_Type,Storage_1_Size_GB,Storage_2_Type,Storage_2_Size_GB,cpu_name,gpu_name
0,0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,Other,...,1,0,128GB SSD,None,SSD,128,None,0,Intel Core i5,Other
1,1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,Other,...,0,0,128GB Flash Storage,None,Flash Storage,128,None,0,Intel Core i5,Intel HD
2,2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,...,0,1,256GB SSD,None,SSD,256,None,0,Intel Core i5,Intel HD
3,3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,Other,...,1,0,512GB SSD,None,SSD,512,None,0,Intel Core i7,AMD
4,4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,Other,...,1,0,256GB SSD,None,SSD,256,None,0,Intel Core i5,Other


### Column dropped

In [51]:
df = df.drop(columns=['Unnamed: 0'])

In [52]:
df.head()

,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,...,Ips,HD,Storage_1,Storage_2,Storage_1_Type,Storage_1_Size_GB,Storage_2_Type,Storage_2_Size_GB,cpu_name,gpu_name
0,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8,128GB SSD,Intel Iris Plus Graphics 640,Other,1.37,...,1,0,128GB SSD,None,SSD,128,None,0,Intel Core i5,Other
1,Apple,Ultrabook,13.30,1440x900,Intel Core i5 1.8GHz,8,128GB Flash Storage,Intel HD Graphics 6000,Other,1.34,...,0,0,128GB Flash Storage,None,Flash Storage,128,None,0,Intel Core i5,Intel HD
2,HP,Notebook,15.60,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8,256GB SSD,Intel HD Graphics 620,No OS,1.86,...,0,1,256GB SSD,None,SSD,256,None,0,Intel Core i5,Intel HD
3,Apple,Ultrabook,15.40,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16,512GB SSD,AMD Radeon Pro 455,Other,1.83,...,1,0,512GB SSD,None,SSD,512,None,0,Intel Core i7,AMD
4,Apple,Ultrabook,13.30,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8,256GB SSD,Intel Iris Plus Graphics 650,Other,1.37,...,1,0,256GB SSD,None,SSD,256,None,0,Intel Core i5,Other


In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1303 entries, 0 to 1302
Data columns (total 22 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Company            1303 non-null   object 
 1   TypeName           1303 non-null   object 
 2   Inches             1303 non-null   float64
 3   ScreenResolution   1303 non-null   object 
 4   Cpu                1303 non-null   object 
 5   Ram                1303 non-null   int64  
 6   Memory             1303 non-null   object 
 7   Gpu                1303 non-null   object 
 8   OpSys              1303 non-null   object 
 9   Weight             1303 non-null   float64
 10  Price              1303 non-null   float64
 11  Touchscreen        1303 non-null   int64  
 12  Ips                1303 non-null   int64  
 13  HD                 1303 non-null   int64  
 14  Storage_1          1303 non-null   object 
 15  Storage_2          208 non-null    object 
 16  Storage_1_Type     1303 

In [54]:
numerical_features = [x for x in df.columns if df[x].dtype in ('int64','float64')]
print(numerical_features)

['Inches', 'Ram', 'Weight', 'Price', 'Touchscreen', 'Ips', 'HD', 'Storage_1_Size_GB']
